# 🔍 Expresiones Regulares (RegEx) para Ciencias - Estudiante
## Clase de Programación en Shell y Python - - Herramientas Computacionales
## Escuela de Ciencias Aplicadas e Ingeniería

---

### 📚 Contenido del Notebook
1. [Introducción a las Expresiones Regulares](#intro)
2. [Wildcards (Comodines)](#wildcards)
3. [Cuantificadores](#quantifiers)
4. [Grupos de Captura](#capture)
5. [Conjuntos de Caracteres](#charsets)
6. [Anclas y Límites](#anchors)
7. [Escapando Caracteres Especiales](#escape)
8. [Tablas de Referencia Rápida](#tables)
9. [Resumen](#summary)

---

<a id='intro'></a>
## 1. 🎯 Introducción a las Expresiones Regulares

Las **expresiones regulares** (también conocidas como *regexp*, *regex* o *grep*) son un lenguaje poderoso para buscar y reemplazar texto. Son fundamentales para:

- 📁 Reformatear archivos de datos entre programas
- 🧬 Procesar secuencias biológicas (ADN, proteínas)
- 📊 Limpiar y transformar datasets
- 🔄 Automatizar tareas repetitivas de manipulación de texto

> **Nota importante:** Las expresiones regulares están integradas en editores de texto, lenguajes de programación (Python, Perl, R), motores de búsqueda y muchas aplicaciones.

### ¿Por qué son importantes en ciencias?

En biología y otras ciencias, frecuentemente necesitamos:
- Convertir formatos de archivos de secuencias
- Extraer información de headers de FASTA
- Reorganizar datos de coordenadas GPS
- Limpiar datos experimentales

### Uso básico en Python

```python
import re

texto = "Agalma elegans es una especie de sifonóforo"
patron = "elegans"

resultado = re.search(patron, texto)
if resultado:
    print(f"Encontrado: '{resultado.group()}'")
```

<a id='wildcards'></a>
## 2. 🃏 Wildcards (Comodines)

Los **wildcards** son caracteres especiales que representan uno o más caracteres en el texto. Son la base de la flexibilidad de las expresiones regulares.

### Tabla de Wildcards Principales

| Wildcard | Significado | Ejemplo |
|:--------:|:------------|:--------|
| `\w` | Cualquier letra, número o guión bajo (A-Z, a-z, 0-9, _) | `\w+` coincide con `Homo123` |
| `\d` | Cualquier dígito (0-9) | `\d{4}` coincide con `2024` |
| `\s` | Cualquier espacio en blanco (espacio, tab, salto de línea) | `\s+` coincide con `"   "` |
| `\t` | Tabulador | Útil para archivos delimitados por tabs |
| `\r` o `\n` | Salto de línea (depende del sistema) | `\n` en Python/Linux, `\r` en Mac antiguo |
| `.` | **Cualquier carácter** excepto salto de línea | `a.b` coincide con `aXb`, `a1b`, `a b` |

### Versiones Negadas (Mayúsculas)

| Wildcard | Significado |
|:--------:|:------------|
| `\W` | Cualquier carácter que **NO** sea letra, número o guión bajo |
| `\D` | Cualquier carácter que **NO** sea dígito |
| `\S` | Cualquier carácter que **NO** sea espacio en blanco |

### Ejemplos de uso

```python
import re

# \w - caracteres de palabra
texto = "Gen_ABC123 tiene secuencia ATGC"
re.findall(r'\w+', texto)  # ['Gen_ABC123', 'tiene', 'secuencia', 'ATGC']

# \d - dígitos
coordenadas = "+40 46'N +014 15'E"
re.findall(r'\d+', coordenadas)  # ['40', '46', '014', '15']

# . (punto) - cualquier carácter
secuencia = "ATG-CGT-AAA"
re.findall(r'A.G', secuencia)  # ['ATG', 'AAA'] - No! solo 'ATG'
```

<a id='quantifiers'></a>
## 3. 🔢 Cuantificadores

Los **cuantificadores** especifican cuántas veces debe aparecer un elemento.

### Tabla de Cuantificadores

| Cuantificador | Significado | Ejemplo | Coincide con |
|:-------------:|:------------|:--------|:-------------|
| `+` | **Uno o más** | `\d+` | `1`, `23`, `456` |
| `*` | **Cero o más** | `\d*` | `""`, `1`, `23` |
| `?` | **Cero o uno** (opcional) | `colou?r` | `color`, `colour` |
| `{n}` | **Exactamente n veces** | `\d{4}` | `2024` (4 dígitos) |
| `{n,}` | **Al menos n veces** | `\d{2,}` | `12`, `123`, `1234` |
| `{n,m}` | **Entre n y m veces** | `\d{2,4}` | `12`, `123`, `1234` |

### ⚠️ Greedy vs Non-Greedy (Codicioso vs No codicioso)

Por defecto, los cuantificadores son **codiciosos** (greedy): intentan coincidir con la mayor cantidad de texto posible.

Agregar `?` después del cuantificador lo hace **no codicioso** (non-greedy): coincide con la menor cantidad posible.

| Greedy | Non-Greedy | Comportamiento |
|:------:|:----------:|:---------------|
| `+` | `+?` | Uno o más (mínimo) |
| `*` | `*?` | Cero o más (mínimo) |
| `{n,m}` | `{n,m}?` | Entre n y m (mínimo) |

### Ejemplo: Problema de greediness

```python
# Secuencia con poly-A al final
secuencia = "CCAAGAGGACAACAAGACATTTAACAAATCACATCTTTGTATTTTTGGTTAGAGTTGAAAAAAA"

# Greedy: .+ captura TODO hasta la última A
re.findall(r'.+A', secuencia)  # Captura casi toda la secuencia

# Non-greedy: .+? captura el mínimo antes de cada A
re.findall(r'.+?A', secuencia)  # Captura segmentos cortos

# Para remover poly-A final, usar ancla $
re.sub(r'A+$', '', secuencia)  # Remueve las A's del final
```

<a id='capture'></a>
## 4. 📦 Grupos de Captura

Los **grupos de captura** son una de las características más poderosas de las expresiones regulares. Permiten:

1. **Extraer** partes específicas del texto encontrado
2. **Reutilizar** el texto capturado en el reemplazo

### Sintaxis

- **Capturar:** Usar paréntesis `( )` alrededor del patrón a capturar
- **Referenciar:** Usar `\1`, `\2`, etc. (o `$1`, `$2` en algunos lenguajes)
  - En Python usamos `\1` en el string de reemplazo, o `.group(1)` para acceder

### Ejemplo Visual

```
Texto:    Agalma elegans
Patrón:   (\w)\w+ (\w+)
          ↓  ↓    ↓
         \1  -   \2
          A      elegans
```

### Ejemplo: Abreviar nombres de especies

```python
especies = """Agalma elegans
Frillagalma vitiazi
Homo sapiens"""

# Patrón: (primera letra)(resto del género) (especie)
patron = r'(\w)\w+ (\w+)'
reemplazo = r'\1. \2'

re.sub(patron, reemplazo, especies)
# Resultado:
# A. elegans
# F. vitiazi
# H. sapiens
```

### Ejemplo: Reformatear headers FASTA

```python
# De: ">CAA58790.1= GFP [Aequorea victoria]"
# A:  ">CAA58790_Aequorea"

header = ">CAA58790.1= GFP [Aequorea victoria]"

patron = r'(>\w+).+\[(\w+).+'
reemplazo = r'\1_\2'

re.sub(patron, reemplazo, header)
# Resultado: ">CAA58790_Aequorea"
```

<a id='charsets'></a>
## 5. 🎨 Conjuntos de Caracteres Personalizados

Los **conjuntos de caracteres** permiten crear wildcards personalizados usando corchetes `[ ]`.

### Sintaxis Básica

| Sintaxis | Significado | Ejemplo |
|:--------:|:------------|:--------|
| `[AGCT]` | Cualquiera de estos caracteres | Nucleótidos de ADN |
| `[A-Z]` | Rango de A a Z (mayúsculas) | Letras mayúsculas |
| `[a-z]` | Rango de a a z (minúsculas) | Letras minúsculas |
| `[0-9]` | Cualquier dígito (igual que `\d`) | Números |
| `[A-Za-z]` | Cualquier letra | Mayúsculas o minúsculas |
| `[^AGCT]` | **Negación**: cualquier carácter EXCEPTO estos | No es nucleótido |
| `[^\t]` | Cualquier carácter excepto tab | Para parsear columnas |

### ⚠️ Caracteres Especiales dentro de [ ]

- El guión `-` define rangos (debe escaparse `\-` si se quiere literal)
- El caret `^` al inicio significa negación
- La mayoría de los caracteres especiales pierden su significado dentro de `[ ]`

### Ejemplo: Validar secuencias de ADN

```python
secuencia = "ATGCGATCGATCG"

# Verificar que solo contenga A, T, G, C
if re.fullmatch(r'[ATGC]+', secuencia):
    print("ADN válido")

# Encontrar caracteres no válidos
secuencia_mala = "ATGCGATXGATCG"
invalidos = re.findall(r'[^ATGC]', secuencia_mala)  # ['X']
```

### Ejemplo: Parsear archivos delimitados por tabs

El patrón `[^\t]+` significa "uno o más caracteres que NO sean tab", perfecto para capturar columnas:

```python
datos = "GenA\t100\t0.95\tAlta expresión"

# Capturar 4 columnas
patron = r'([^\t]+)\t([^\t]+)\t([^\t]+)\t([^\t]+)'
match = re.search(patron, datos)

gen, valor, score, nota = match.groups()
# gen='GenA', valor='100', score='0.95', nota='Alta expresión'

# Reordenar columnas
re.sub(patron, r'\3\t\1\t\2', datos)
# Resultado: "0.95\tGenA\t100"
```

<a id='anchors'></a>
## 6. ⚓ Anclas y Límites

Las **anclas** no coinciden con caracteres, sino con **posiciones** en el texto.

### Tabla de Anclas

| Ancla | Significado | Ejemplo de uso |
|:-----:|:------------|:---------------|
| `^` | **Inicio de línea** | `^>` encuentra `>` solo al inicio |
| `$` | **Fin de línea** | `\d+$` encuentra números al final |
| `\b` | **Límite de palabra** | `\bgen\b` encuentra "gen" pero no "genoma" |
| `\B` | **NO límite de palabra** | `\Bgen` encuentra "gen" dentro de palabras |

### ⚠️ Nota sobre `^`

El caret `^` tiene **dos significados** según el contexto:
- **Fuera de `[ ]`**: Inicio de línea
- **Dentro de `[ ]` al inicio**: Negación

### Ejemplo: Encontrar headers en FASTA

```python
fasta = """>seq1 Homo sapiens
ATGCGATCGATCG
>seq2 Mus musculus
ATGCGATCGATCGATCG"""

# Encontrar líneas que empiezan con >
headers = re.findall(r'^>.*', fasta, re.MULTILINE)
# ['>seq1 Homo sapiens', '>seq2 Mus musculus']

# Agregar texto al final de cada header
re.sub(r'^(>.*)$', r'\1 [PROCESADO]', fasta, flags=re.MULTILINE)
```

### Ejemplo: Reemplazar solo en primera columna

```python
datos = """-1\t23.5\t-1.2
ABC\t-1\t45.6
-1\t12.3\t78.9"""

# Reemplazar -1 SOLO al inicio de línea (primera columna)
re.sub(r'^-1\t', 'NaN\t', datos, flags=re.MULTILINE)
# Solo cambia el -1 de la primera columna, no el de otras columnas
```

<a id='escape'></a>
## 7. 🔧 Escapando Caracteres Especiales

Muchos caracteres tienen significado especial en regex. Para buscarlos literalmente, se deben **escapar** con `\`.

### Caracteres que Necesitan Escape

| Carácter | Significado especial | Para búsqueda literal |
|:--------:|:---------------------|:---------------------:|
| `.` | Cualquier carácter | `\.` |
| `*` | Cero o más | `\*` |
| `+` | Uno o más | `\+` |
| `?` | Cero o uno / non-greedy | `\?` |
| `^` | Inicio de línea | `\^` |
| `$` | Fin de línea | `\$` |
| `( )` | Grupo de captura | `\( \)` |
| `[ ]` | Conjunto de caracteres | `\[ \]` |
| `{ }` | Cuantificador | `\{ \}` |
| `\` | Escape | `\\` |
| `|` | Alternativa (OR) | `\|` |

### Ejemplo: Extraer autor de nombre científico

```python
especie = "Physalia physalis (Linnaeus)"

# Los paréntesis deben escaparse para buscarlos literalmente
# pero usamos ( ) sin escapar para CAPTURAR
patron = r'(\w+) (\w+) \((\w+)\)'
#                      \(   \)  <- Paréntesis literales escapados
#          (  )  (  )  (   )    <- Grupos de captura

match = re.search(patron, especie)
genero, especie_nombre, autor = match.groups()
# genero='Physalia', especie_nombre='physalis', autor='Linnaeus'

# Reformatear
re.sub(patron, r'\1_\2_\3', especie)
# Resultado: "Physalia_physalis_Linnaeus"
```

<a id='tables'></a>
## 8. 📋 Tablas de Referencia Rápida

### Wildcards y Caracteres Especiales

| Símbolo | Descripción | Ejemplo |
|:-------:|:------------|:--------|
| `\w` | Letra, dígito o guión bajo | `\w+` → `Gene_123` |
| `\W` | NO es letra, dígito o guión bajo | `\W+` → `!@#` |
| `\d` | Dígito (0-9) | `\d{4}` → `2024` |
| `\D` | NO es dígito | `\D+` → `abc` |
| `\s` | Espacio en blanco | `\s+` → `   ` |
| `\S` | NO es espacio en blanco | `\S+` → `texto` |
| `\t` | Tabulador | Para TSV |
| `\n` | Nueva línea | Fin de línea |
| `.` | Cualquier carácter (excepto \n) | `a.b` → `aXb` |

### Cuantificadores

| Símbolo | Descripción | Greedy | Non-greedy |
|:-------:|:------------|:------:|:----------:|
| `*` | Cero o más | `.*` | `.*?` |
| `+` | Uno o más | `.+` | `.+?` |
| `?` | Cero o uno | `a?` | - |
| `{n}` | Exactamente n | `\d{3}` | - |
| `{n,}` | n o más | `\d{2,}` | `\d{2,}?` |
| `{n,m}` | Entre n y m | `\d{2,4}` | `\d{2,4}?` |

### Anclas y Límites

| Símbolo | Descripción |
|:-------:|:------------|
| `^` | Inicio de línea |
| `$` | Fin de línea |
| `\b` | Límite de palabra |
| `\B` | NO es límite de palabra |

### Conjuntos de Caracteres

| Sintaxis | Descripción |
|:--------:|:------------|
| `[abc]` | a, b, o c |
| `[^abc]` | NO es a, b, ni c |
| `[a-z]` | Rango de a a z |
| `[A-Za-z0-9]` | Letra o dígito |

### Grupos y Referencias

| Sintaxis | Descripción | En Python |
|:--------:|:------------|:----------|
| `(...)` | Grupo de captura | `.group(1)` |
| `\1`, `\2` | Referencia al grupo | En `re.sub()` |
| `(?:...)` | Grupo sin captura | No crea referencia |

### Funciones Principales del Módulo `re` en Python

| Función | Descripción | Retorna |
|:--------|:------------|:--------|
| `re.search(patron, texto)` | Encuentra primera coincidencia | `Match` o `None` |
| `re.match(patron, texto)` | Coincide solo al inicio | `Match` o `None` |
| `re.findall(patron, texto)` | Encuentra todas las coincidencias | Lista |
| `re.finditer(patron, texto)` | Iterador de coincidencias | Iterador |
| `re.sub(patron, reemplazo, texto)` | Buscar y reemplazar | String modificado |
| `re.split(patron, texto)` | Dividir por patrón | Lista |
| `re.fullmatch(patron, texto)` | Coincidir string completo | `Match` o `None` |

### Flags Comunes

| Flag | Descripción |
|:-----|:------------|
| `re.IGNORECASE` o `re.I` | Ignorar mayúsculas/minúsculas |
| `re.MULTILINE` o `re.M` | `^` y `$` coinciden en cada línea |
| `re.DOTALL` o `re.S` | `.` también coincide con `\n` |

### Operaciones Comunes con Ejemplos

| Operación | Find | Replace | Resultado |
|:----------|:-----|:--------|:----------|
| Separar `nano_128.dat` | `(\w+)_(\d+)\.(\w+)` | `\1\t\2\t\3` | `nano  128  dat` |
| Intercambiar columnas | `(\w+)\t(\w+)` | `\2\t\1` | Columnas invertidas |
| Unir líneas | `\n` | ` ` | Todo en una línea |
| Separar por coma | `,` | `\n` | Cada elemento en línea |
| Abreviar género | `(\w)\w+ (\w+)` | `\1. \2` | `A. elegans` |
| Eliminar hasta X | `^.*X` | `` | Desde inicio hasta X |
| Eliminar desde X | `X.*$` | `` | Desde X hasta final |

<a id='summary'></a>
## 9. 📝 Resumen

### Conceptos Clave Aprendidos

1. **Wildcards básicos**:
   - `\w` (palabra), `\d` (dígito), `\s` (espacio), `.` (cualquier carácter)
   - Sus versiones negadas: `\W`, `\D`, `\S`

2. **Cuantificadores**:
   - `+` (uno o más), `*` (cero o más), `?` (cero o uno)
   - `{n}`, `{n,}`, `{n,m}` para control preciso
   - Greedy vs non-greedy (`+?`, `*?`)

3. **Grupos de captura**:
   - Usar `( )` para capturar partes del patrón
   - Referenciar con `\1`, `\2`, etc. en reemplazos

4. **Conjuntos de caracteres**:
   - `[abc]` para caracteres específicos
   - `[a-z]` para rangos
   - `[^abc]` para negación

5. **Anclas**:
   - `^` (inicio de línea), `$` (fin de línea)
   - `\b` (límite de palabra)

6. **Escape de caracteres especiales**:
   - Usar `\` antes de `.`, `*`, `+`, `?`, `(`, `)`, `[`, `]`, `{`, `}`, `^`, `$`, `\`, `|`

### Mejores Prácticas

1. **Construir patrones incrementalmente**: Empieza simple y agrega complejidad
2. **Probar con datos de ejemplo**: Antes de aplicar a archivos grandes
3. **Usar raw strings en Python**: `r'patrón'` para evitar problemas con `\`
4. **Documentar patrones complejos**: Los regex pueden ser difíciles de leer después
5. **Validar resultados**: Verificar el número de coincidencias esperadas
6. **Sobrespecificar cuando sea crítico**: Mejor que falle completamente a que haga cambios incorrectos

### Estrategia para Construir Patrones Complejos

1. **Copiar** una línea de ejemplo del texto
2. **Identificar** qué partes quieres capturar
3. **Marcar** con paréntesis las capturas
4. **Reemplazar** texto literal por wildcards
5. **Agregar** cuantificadores donde sea necesario
6. **Probar** con Find antes de Replace
7. **Verificar** el número de coincidencias

---

## 📚 Referencias y Recursos Adicionales

- **Libro base**: Haddock, S.H.D. & Dunn, C.W. (2011). *Practical Computing for Biologists*. Sinauer Associates. Capítulos 2 y 3.
- **Documentación de Python**: [re — Regular expression operations](https://docs.python.org/3/library/re.html)
- **Práctica online**: [regex101.com](https://regex101.com/) - Excelente para probar patrones con explicación visual
- **Tutorial interactivo**: [RegexOne](https://regexone.com/) - Ejercicios progresivos
- **Cheat sheet**: [Regular Expressions Cheat Sheet](https://www.rexegg.com/regex-quickstart.html)

---

*Material basado en "Practical Computing for Biologists" - Adaptado para el curso de Programación en Shell y Python, Ciencias, Tercer Semestre*